In [14]:
import pandas as pd
import nltk
import spacy
import re

from nltk.corpus import stopwords
from collections import Counter

In [15]:
# 1. Download NLTK Stopwords
nltk.download("stopwords")

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\hp\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [16]:
# 2. Load Dataset
INPUT_FILE = "fakenews_stemming_lemmatization.csv"
OUTPUT_FILE = "fakenews_stopwords_comparison.csv"

df = pd.read_csv(INPUT_FILE)

print("Dataset loaded successfully!")
print("Dataset shape:", df.shape)

Dataset loaded successfully!
Dataset shape: (2212, 10)


In [17]:
# 3. Select Text Column

if "spacy_lemmatized_text" in df.columns:
    TEXT_COLUMN = "spacy_lemmatized_text"

elif "body_text_preprocessed" in df.columns:
    TEXT_COLUMN = "body_text_preprocessed"

elif "headline_preprocessed" in df.columns:
    TEXT_COLUMN = "headline_preprocessed"

else:
    raise ValueError(
        "No suitable text column found. "
        "Run the previous preprocessing tasks first."
    )

print("Selected text column:", TEXT_COLUMN)

Selected text column: spacy_lemmatized_text


In [18]:

# 4. Initialize spaCy
try:
    nlp = spacy.load("en_core_web_sm")

except OSError:
    raise RuntimeError(
        "spaCy English model is missing. Run:\n"
        "python -m spacy download en_core_web_sm"
    )

In [19]:
# 5. Load Predefined Stopword Lists
nltk_stopwords = set(stopwords.words("english"))

spacy_stopwords = set(nlp.Defaults.stop_words)

combined_stopwords = (
    nltk_stopwords | spacy_stopwords
)

In [20]:
# 6. Protect Important Words
# These words can be important for sentiment,
# claims, and fake-news classification.

protected_words = {
    "no",
    "not",
    "nor",
    "never",
    "neither",
    "without",
    "against",
    "fake",
    "false",
    "fraud",
    "allegedly",
    "claim",
    "claims"
}


# Remove protected words from every stopword list.

nltk_stopwords -= protected_words
spacy_stopwords -= protected_words
combined_stopwords -= protected_words


In [21]:
# 7. Analyze Frequent Dataset Words
def tokenize(text):

    if pd.isna(text):
        return []

    return re.findall(
        r"\b[a-z]+\b",
        str(text).lower()
    )


word_frequency = Counter()

for text in df[TEXT_COLUMN].fillna(""):

    word_frequency.update(tokenize(text))


total_tokens = sum(word_frequency.values())

print("\nTotal tokens:", total_tokens)
print("Unique words:", len(word_frequency))



Total tokens: 22257
Unique words: 82


In [22]:

# 8. Identify Dataset-Specific Candidates

# Candidate words must:
# 1. Occur frequently in the dataset.
# 2. Be reasonably short.
# 3. Not belong to the protected-word list.
# 4. Not already be in the combined stopword list.

MIN_FREQUENCY = 100
MAX_WORD_LENGTH = 3

dataset_candidates = {
    word
    for word, frequency in word_frequency.items()
    if frequency >= MIN_FREQUENCY
    and len(word) <= MAX_WORD_LENGTH
    and word not in combined_stopwords
    and word not in protected_words
}

print("\nDataset-specific stopword candidates:")
print(sorted(dataset_candidates))


Dataset-specific stopword candidates:
[]


In [23]:
# 9. Create Custom Stopword Strategy
custom_stopwords = (
    combined_stopwords | dataset_candidates
)

# Always preserve protected words.

custom_stopwords -= protected_words

In [24]:
# 10. Stopword Removal Function
def remove_stopwords(text, stopword_list):

    words = tokenize(text)

    filtered_words = [
        word
        for word in words
        if word not in stopword_list
    ]

    return " ".join(filtered_words)

In [25]:
# 11. Apply Multiple Strategies
print("\nApplying NLTK stopwords...")

df["nltk_stopword_text"] = df[TEXT_COLUMN].apply(
    lambda text: remove_stopwords(
        text, nltk_stopwords
    )
)

print("Applying spaCy stopwords...")

df["spacy_stopword_text"] = df[TEXT_COLUMN].apply(
    lambda text: remove_stopwords(
        text, spacy_stopwords
    )
)

print("Applying combined stopwords...")

df["combined_stopword_text"] = df[TEXT_COLUMN].apply(
    lambda text: remove_stopwords(
        text, combined_stopwords
    )
)

print("Applying custom stopwords...")

df["custom_stopword_text"] = df[TEXT_COLUMN].apply(
    lambda text: remove_stopwords(
        text, custom_stopwords
    )
)


Applying NLTK stopwords...
Applying spaCy stopwords...
Applying combined stopwords...
Applying custom stopwords...


In [26]:
# 12. Vocabulary Analysis
def get_vocabulary(text_series):

    vocabulary = set()

    for text in text_series.fillna(""):

        vocabulary.update(tokenize(text))

    return vocabulary


original_vocab = get_vocabulary(
    df[TEXT_COLUMN]
)


def calculate_reduction(original_vocab, new_vocab):

    if not original_vocab:
        return 0.0

    return (
        len(original_vocab - new_vocab)
        / len(original_vocab)
    ) * 100


strategies = {
    "Original": TEXT_COLUMN,
    "NLTK": "nltk_stopword_text",
    "spaCy": "spacy_stopword_text",
    "Combined": "combined_stopword_text",
    "Custom": "custom_stopword_text"
}

comparison_results = []

for strategy, column in strategies.items():

    vocab = get_vocabulary(df[column])

    comparison_results.append({
        "Strategy": strategy,
        "Vocabulary Size": len(vocab),
        "Vocabulary Reduction (%)":
            calculate_reduction(
                original_vocab, vocab
            )
    })


comparison_df = pd.DataFrame(
    comparison_results
)

print("\nVocabulary Comparison:")
print(
    comparison_df.to_string(index=False)
)


Vocabulary Comparison:
Strategy  Vocabulary Size  Vocabulary Reduction (%)
Original               82                  0.000000
    NLTK               60                 26.829268
   spaCy               51                 37.804878
Combined               51                 37.804878
  Custom               51                 37.804878


In [27]:
# 13. Compare Token Retention
print("\nToken Retention Comparison:")

original_token_count = sum(
    len(tokenize(text))
    for text in df[TEXT_COLUMN].fillna("")
)

for strategy, column in strategies.items():

    remaining_tokens = sum(
        len(tokenize(text))
        for text in df[column].fillna("")
    )

    if original_token_count > 0:

        retention = (
            remaining_tokens
            / original_token_count
        ) * 100

    else:
        retention = 0.0

    print(
        f"{strategy}: "
        f"{remaining_tokens} tokens remaining, "
        f"{retention:.2f}% retained"
    )


Token Retention Comparison:
Original: 22257 tokens remaining, 100.00% retained
NLTK: 13672 tokens remaining, 61.43% retained
spaCy: 11712 tokens remaining, 52.62% retained
Combined: 11712 tokens remaining, 52.62% retained
Custom: 11712 tokens remaining, 52.62% retained


In [28]:
# 14. Display Example Results
print("\nExample Comparison:")

for index in range(min(3, len(df))):

    print("\nOriginal:")
    print(df[TEXT_COLUMN].iloc[index])

    print("\nNLTK:")
    print(df["nltk_stopword_text"].iloc[index])

    print("\nspaCy:")
    print(df["spacy_stopword_text"].iloc[index])

    print("\nCombined:")
    print(df["combined_stopword_text"].iloc[index])

    print("\nCustom:")
    print(df["custom_stopword_text"].iloc[index])



Example Comparison:

Original:
the prime minister attend the inauguration ceremony as the chief guest

NLTK:
prime minister attend inauguration ceremony chief guest

spaCy:
prime minister attend inauguration ceremony chief guest

Combined:
prime minister attend inauguration ceremony chief guest

Custom:
prime minister attend inauguration ceremony chief guest

Original:
government source state that an official announcement will be make soon

NLTK:
government source state official announcement make soon

spaCy:
government source state official announcement soon

Combined:
government source state official announcement soon

Custom:
government source state official announcement soon

Original:
local administration have visit the spot for further inspection

NLTK:
local administration visit spot inspection

spaCy:
local administration visit spot inspection

Combined:
local administration visit spot inspection

Custom:
local administration visit spot inspection


In [29]:

# 15. Saving Results
df.to_csv(OUTPUT_FILE, index=False)

comparison_df.to_csv(
    "T9_stopword_vocabulary_comparison.csv",
    index=False
)

pd.DataFrame({
    "Dataset_Specific_Candidates":
        sorted(dataset_candidates)
}).to_csv(
    "T9_dataset_stopword_candidates.csv",
    index=False
)


print("\nStopword removal completed successfully!")

print("Processed dataset:", OUTPUT_FILE)

print(
    "Vocabulary report:",
    "T9_stopword_vocabulary_comparison.csv"
)

print(
    "Dataset-specific candidates:",
    "T9_dataset_stopword_candidates.csv"
)


Stopword removal completed successfully!
Processed dataset: fakenews_stopwords_comparison.csv
Vocabulary report: T9_stopword_vocabulary_comparison.csv
Dataset-specific candidates: T9_dataset_stopword_candidates.csv
